# LeetCode 1619 — Mean of Array After Removing Some Elements bằng Python

**Link đề:** https://leetcode.com/problems/mean-of-array-after-removing-some-elements/
**Độ khó:** Easy · **Chủ đề:** Array, Sorting

```python
class Solution:
    def trimMean(self, arr: list[int]) -> float:
        ...
```

## Mục tiêu học
1. Hiểu **trung bình cắt (trimmed mean)**: bỏ 5% nhỏ nhất và 5% lớn nhất rồi lấy trung bình — một khái niệm thống kê thực tế.
2. Tính đúng **số phần tử cần bỏ** `k = n // 20` và vì sao đề cho `n` chia hết cho 20.
3. Ôn `sorted`, slicing `arr[k:n-k]`, `sum`, phép chia `/` cho số thực, so sánh số thực với sai số (`math.isclose`).
4. Viết 4 cách: sắp xếp + cắt → sắp xếp + trừ phần cắt → `heapq.nsmallest/nlargest` → đếm tần suất (counting); so sánh độ phức tạp.
5. Trace, test có dung sai `1e-5`, và mở rộng sang trung bình cắt theo % bất kỳ, trung vị.

## 1. Đề bài

Cho mảng số nguyên `arr`. Trả về **trung bình cộng** của các số còn lại sau khi **bỏ 5% số nhỏ nhất** và **5% số lớn nhất**.

Đáp án sai lệch không quá `10^-5` so với đáp án đúng được chấp nhận.

| `arr` (n) | Kết quả | Ghi chú |
|---|:---:|---|
| `[1,2,2,...,2,3]` (n=20) | `2.00000` | bỏ 1 nhỏ nhất (`1`) và 1 lớn nhất (`3`) → còn toàn `2` |
| `[6,2,7,5,1,2,0,3,10,2,5,0,5,5,0,8,7,6,8,0]` (n=20) | `4.00000` | |
| mảng 40 phần tử (ví dụ 3) | `4.77778` | bỏ 2 nhỏ nhất, 2 lớn nhất |

**Constraints:** `20 <= arr.length <= 1000`; `arr.length` **chia hết cho 20**; `0 <= arr[i] <= 10^5`.

In [1]:
ex1 = [1] + [2] * 18 + [3]
ex2 = [6, 2, 7, 5, 1, 2, 0, 3, 10, 2, 5, 0, 5, 5, 0, 8, 7, 6, 8, 0]
ex3 = [6, 0, 7, 0, 7, 5, 7, 8, 3, 4, 0, 7, 8, 1, 6, 8, 1, 1, 2, 4, 8, 1, 9, 5, 4, 3, 8, 5, 10, 8, 6, 6, 1, 0, 6, 10, 8, 2, 3, 4]
examples = [(ex1, 2.0), (ex2, 4.0), (ex3, 4.77778)]
for arr, expected in examples:
    print("n =", len(arr), "| bỏ mỗi đầu", len(arr) // 20, "phần tử | kỳ vọng", expected)

n = 20 | bỏ mỗi đầu 1 phần tử | kỳ vọng 2.0
n = 20 | bỏ mỗi đầu 1 phần tử | kỳ vọng 4.0
n = 40 | bỏ mỗi đầu 2 phần tử | kỳ vọng 4.77778


## 2. Nền tảng: 5% là bao nhiêu phần tử?

5% của `n` là `n * 5 / 100 = n / 20`. Vì `n` **chia hết cho 20**, `n / 20` luôn là số nguyên → không phải lo làm tròn. Ta dùng chia nguyên `k = n // 20`.

| n | k = n // 20 | Còn lại n − 2k |
|:-:|:-:|:-:|
| 20 | 1 | 18 |
| 40 | 2 | 36 |
| 1000 | 50 | 900 |

In [2]:
for n in [20, 40, 100, 1000]:
    print(f"n={n:>4}: 5% = {n * 5 / 100} | n // 20 = {n // 20} | còn lại {n - 2 * (n // 20)}")

n=  20: 5% = 1.0 | n // 20 = 1 | còn lại 18
n=  40: 5% = 2.0 | n // 20 = 2 | còn lại 36
n= 100: 5% = 5.0 | n // 20 = 5 | còn lại 90
n=1000: 5% = 50.0 | n // 20 = 50 | còn lại 900


### 2.1. `/` và `//` khác nhau

- `a / b` luôn trả **số thực** (`float`): `7 / 2 = 3.5`, `6 / 2 = 3.0`.
- `a // b` trả **phần nguyên** (chia nguyên, làm tròn xuống).

Đề trả về `float` → phép tính trung bình cuối cùng dùng `/`. Số lượng phần tử bỏ đi dùng `//`.

In [3]:
print(7 / 2, 6 / 2, type(6 / 2))
print(7 // 2, 6 // 2, type(6 // 2))

3.5 3.0 <class 'float'>
3 3 <class 'int'>


### 2.2. `sorted` và slicing `a[k:n-k]` — bỏ hai đầu

Sau khi sắp tăng dần, `k` phần tử nhỏ nhất nằm ở **đầu**, `k` phần tử lớn nhất ở **cuối**. `a[k:n-k]` lấy phần **giữa**. Có thể viết `a[k:-k]` nhưng cẩn thận: nếu `k = 0` thì `a[0:-0]` = `a[0:0]` = **rỗng**! (Bài này `k >= 1` nên không gặp, nhưng nên biết.)

In [4]:
a = sorted([5, 1, 9, 3, 7, 2])
n, k = len(a), 1
print("sorted:", a)
print("a[k:n-k] =", a[k:n - k])
print("a[k:-k]  =", a[k:-k])
print("cẩn thận k=0: a[0:-0] =", a[0:-0], "| a[0:n-0] =", a[0:n - 0])

sorted: [1, 2, 3, 5, 7, 9]
a[k:n-k] = [2, 3, 5, 7]
a[k:-k]  = [2, 3, 5, 7]
cẩn thận k=0: a[0:-0] = [] | a[0:n-0] = [1, 2, 3, 5, 7, 9]


### 2.3. So sánh số thực: dùng sai số, đừng dùng `==`

Số thực trong máy tính có sai số làm tròn: `0.1 + 0.2 != 0.3`. Khi kiểm tra kết quả, dùng `abs(a - b) < 1e-5` hoặc `math.isclose(a, b, abs_tol=1e-5)` — đúng tinh thần "sai lệch không quá `10^-5`" của đề.

In [5]:
import math
print(0.1 + 0.2, 0.1 + 0.2 == 0.3)
print(abs((0.1 + 0.2) - 0.3) < 1e-5, math.isclose(0.1 + 0.2, 0.3, abs_tol=1e-5))
print("làm tròn để in:", round(4.777777777, 5), f"{4.777777777:.5f}")

0.30000000000000004 False
True True
làm tròn để in: 4.77778 4.77778


## 3. Cách 1 — Sắp xếp, cắt hai đầu, lấy trung bình (đúng hint)

1. `a = sorted(arr)`.
2. `k = len(arr) // 20`.
3. `middle = a[k:n-k]`.
4. Trả `sum(middle) / len(middle)`.

In [6]:
def trim_mean_slice(arr: list[int]) -> float:
    n = len(arr)
    k = n // 20
    a = sorted(arr)
    middle = a[k:n - k]
    return sum(middle) / len(middle)

for arr, expected in examples:
    got = trim_mean_slice(arr)
    print(f"{got:.5f} | kỳ vọng {expected} | khớp? {abs(got - expected) < 1e-5}")

2.00000 | kỳ vọng 2.0 | khớp? True
4.00000 | kỳ vọng 4.0 | khớp? True
4.77778 | kỳ vọng 4.77778 | khớp? True


### 3.1. Dùng thư viện `statistics`

Thư viện chuẩn `statistics` có `mean` (trung bình). Gọn hơn nhưng chậm hơn `sum/len` một chút; vẫn hợp lệ trên LeetCode.

In [7]:
import statistics
a = sorted(ex2)
k = len(a) // 20
print(statistics.mean(a[k:len(a) - k]), "| sum/len:", sum(a[k:len(a) - k]) / (len(a) - 2 * k))

4 | sum/len: 4.0


## 4. Cách 2 — Sắp xếp rồi **trừ** phần bị cắt (không tạo slice)

`sum(middle) = sum(arr) - sum(k nhỏ nhất) - sum(k lớn nhất)`. Số phần tử còn lại là `n - 2k` = `0.9 * n`.

Cách này không tạo list `middle` mới — ý tưởng "**tổng toàn bộ trừ phần bỏ đi**" rất hay dùng lại.

In [8]:
def trim_mean_subtract(arr: list[int]) -> float:
    n = len(arr)
    k = n // 20
    a = sorted(arr)
    total = sum(a) - sum(a[:k]) - sum(a[n - k:])
    return total / (n - 2 * k)

print([round(trim_mean_subtract(a), 5) for a, _ in examples])

[2.0, 4.0, 4.77778]


## 5. Nút thắt và insight — có cần sắp xếp **toàn bộ**?

Sắp xếp tốn `O(n log n)`. Nhưng ta chỉ cần biết **`k` số nhỏ nhất** và **`k` số lớn nhất** (k = 5% n), không cần thứ tự của 90% ở giữa!

**Insight 1:** `heapq.nsmallest(k, arr)` / `heapq.nlargest(k, arr)` lấy `k` phần tử nhỏ/lớn nhất trong `O(n log k)` bằng **heap** (đống).

**Insight 2:** giá trị bị giới hạn `0..10^5` → **đếm tần suất** rồi đi từ giá trị nhỏ lên/lớn xuống để "bỏ" `k` phần tử mỗi đầu.

### 5.1. `heapq` là gì?

**Heap (min-heap)** là cấu trúc cho phép lấy phần tử nhỏ nhất nhanh (`O(log n)` mỗi lần lấy). Module `heapq` cung cấp:
- `heapq.nsmallest(k, iterable)` → list `k` phần tử nhỏ nhất (đã sắp tăng).
- `heapq.nlargest(k, iterable)` → list `k` phần tử lớn nhất (đã sắp giảm).

In [9]:
import heapq
data = [6, 2, 7, 5, 1, 2, 0, 3]
print("2 nhỏ nhất:", heapq.nsmallest(2, data))
print("2 lớn nhất:", heapq.nlargest(2, data))
print("gốc không đổi:", data)

2 nhỏ nhất: [0, 1]
2 lớn nhất: [7, 6]
gốc không đổi: [6, 2, 7, 5, 1, 2, 0, 3]


In [10]:
def trim_mean_heap(arr: list[int]) -> float:
    n = len(arr)
    k = n // 20
    removed = sum(heapq.nsmallest(k, arr)) + sum(heapq.nlargest(k, arr))
    return (sum(arr) - removed) / (n - 2 * k)

print([round(trim_mean_heap(a), 5) for a, _ in examples])

[2.0, 4.0, 4.77778]


Chú ý: nếu một giá trị xuất hiện nhiều lần, `nsmallest` chỉ lấy **đúng `k` phần tử** (đúng như cắt trong mảng đã sắp) — ví dụ 2 có bốn số `0` mà `k = 1` thì chỉ bỏ **một** số `0`.

## 6. Cách 4 — Đếm tần suất (counting)

Dùng `collections.Counter` đếm số lần xuất hiện mỗi giá trị, rồi:
- Đi từ giá trị **nhỏ** lên, "tiêu" `k` phần tử, cộng dồn tổng bị bỏ.
- Đi từ giá trị **lớn** xuống, tương tự.

`Counter(arr)` là dict `giá_trị → số_lần`. `sorted(counter)` cho các giá trị khác nhau tăng dần.

In [11]:
from collections import Counter
c = Counter([0, 5, 0, 5, 5, 2])
print(c, "| giá trị tăng dần:", sorted(c), "| c[5] =", c[5])

def removed_sum(values_in_order, counter, k):
    # Bỏ k phần tử theo thứ tự values_in_order, trả về tổng bị bỏ
    total, left = 0, k
    for v in values_in_order:
        take = min(left, counter[v])
        total += take * v
        left -= take
        if left == 0:
            break
    return total

print("bỏ 3 nhỏ nhất:", removed_sum(sorted(c), c, 3), "(0+0+2)")
print("bỏ 2 lớn nhất:", removed_sum(sorted(c, reverse=True), c, 2), "(5+5)")

Counter({5: 3, 0: 2, 2: 1}) | giá trị tăng dần: [0, 2, 5] | c[5] = 3
bỏ 3 nhỏ nhất: 2 (0+0+2)
bỏ 2 lớn nhất: 10 (5+5)


In [12]:
def trim_mean_counting(arr: list[int]) -> float:
    n = len(arr)
    k = n // 20
    c = Counter(arr)
    keys = sorted(c)                      # số giá trị khác nhau, thường << n
    low = removed_sum(keys, c, k)
    high = removed_sum(reversed(keys), c, k)
    return (sum(arr) - low - high) / (n - 2 * k)

print([round(trim_mean_counting(a), 5) for a, _ in examples])

[2.0, 4.0, 4.77778]


`reversed(keys)` duyệt list theo chiều ngược mà không tạo list mới. (Cách này vẫn sắp xếp các **giá trị khác nhau** `d` — `O(d log d)`; nếu muốn tuyệt đối `O(n + V)` có thể dùng mảng đếm dài `10^5 + 1`, nhưng trên thực tế chậm hơn với `n ≤ 1000`.)

## 7. TRACE từng bước

Ví dụ 2: `arr = [6,2,7,5,1,2,0,3,10,2,5,0,5,5,0,8,7,6,8,0]`, `n = 20`, `k = 1`.

| Bước | Giá trị |
|---|---|
| sorted | `[0,0,0,0,1,2,2,2,3,5,5,5,5,6,6,7,7,8,8,10]` |
| bỏ k=1 nhỏ nhất | `0` |
| bỏ k=1 lớn nhất | `10` |
| tổng toàn bộ | `82` |
| tổng còn lại | `82 − 0 − 10 = 72` |
| số phần tử còn | `18` |
| trung bình | `72 / 18 = 4.0` |

In [13]:
def trace_trim_mean(arr):
    n = len(arr)
    k = n // 20
    a = sorted(arr)
    print("n =", n, "| k =", k)
    print("sorted =", a)
    print("bỏ đầu:", a[:k], "| bỏ cuối:", a[n - k:])
    middle = a[k:n - k]
    print("giữ lại", len(middle), "phần tử, tổng =", sum(middle))
    ans = sum(middle) / len(middle)
    print(f"trung bình = {sum(middle)} / {len(middle)} = {ans:.5f}")
    return ans

trace_trim_mean(ex2)

n = 20 | k = 1
sorted = [0, 0, 0, 0, 1, 2, 2, 2, 3, 5, 5, 5, 5, 6, 6, 7, 7, 8, 8, 10]
bỏ đầu: [0] | bỏ cuối: [10]
giữ lại 18 phần tử, tổng = 72
trung bình = 72 / 18 = 4.00000


4.0

Trace ví dụ 3 (n = 40, k = 2):

In [14]:
trace_trim_mean(ex3)

n = 40 | k = 2
sorted = [0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 6, 6, 6, 6, 6, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 9, 10, 10]
bỏ đầu: [0, 0] | bỏ cuối: [10, 10]
giữ lại 36 phần tử, tổng = 172
trung bình = 172 / 36 = 4.77778


4.777777777777778

## 8. Code nộp LeetCode

### 8.1. Cách chính — sắp xếp + cắt (ngắn, rõ)

In [15]:
class Solution:
    def trimMean(self, arr: list[int]) -> float:
        n = len(arr)
        k = n // 20
        a = sorted(arr)
        return sum(a[k:n - k]) / (n - 2 * k)

for arr, expected in examples:
    assert abs(Solution().trimMean(arr) - expected) < 1e-5
print("Solution PASS ví dụ đề")

Solution PASS ví dụ đề


### 8.2. Các cách khác

In [16]:
class SolutionHeap:
    def trimMean(self, arr: list[int]) -> float:
        n, k = len(arr), len(arr) // 20
        removed = sum(heapq.nsmallest(k, arr)) + sum(heapq.nlargest(k, arr))
        return (sum(arr) - removed) / (n - 2 * k)

class SolutionCounting:
    def trimMean(self, arr: list[int]) -> float:
        n, k = len(arr), len(arr) // 20
        cnt = Counter(arr)
        keys = sorted(cnt)
        def drop(order):
            left, s = k, 0
            for v in order:
                t = min(left, cnt[v])
                s += t * v
                left -= t
                if left == 0:
                    break
            return s
        return (sum(arr) - drop(keys) - drop(reversed(keys))) / (n - 2 * k)

class SolutionStatistics:
    def trimMean(self, arr: list[int]) -> float:
        k = len(arr) // 20
        a = sorted(arr)
        return statistics.mean(a[k:len(a) - k])

for cls in [SolutionHeap, SolutionCounting, SolutionStatistics]:
    print(cls.__name__, [round(cls().trimMean(a), 5) for a, _ in examples])

SolutionHeap [2.0, 4.0, 4.77778]
SolutionCounting [2.0, 4.0, 4.77778]
SolutionStatistics [2, 4, 4.77778]


Trong `SolutionCounting` có **hàm lồng** `drop` định nghĩa bên trong method — nó "nhìn thấy" biến `k`, `cnt` của hàm ngoài (gọi là **closure**). Tiện để tránh truyền nhiều tham số.

## 9. Vì sao đúng? (invariant)

- Sau khi sắp tăng dần, với mọi `i < j` có `a[i] <= a[j]`. Do đó `a[0..k-1]` là **một** tập `k` phần tử nhỏ nhất và `a[n-k..n-1]` là **một** tập `k` phần tử lớn nhất (khi có giá trị trùng, chọn phần tử nào cũng cho **cùng tổng**).
- Vì `k = n/20 >= 1` và `n - 2k = 0.9n > 0`, phần giữa không rỗng → không chia cho 0.
- Cách trừ: `sum(giữa) = sum(tất cả) − sum(đầu) − sum(cuối)` — đẳng thức của phép cộng.
- Cách heap/đếm: `nsmallest`/`removed_sum` cũng trả về tổng của `k` phần tử nhỏ nhất (tính cả lặp) → cùng tổng với cách sắp xếp.

## 10. Độ phức tạp

Gọi `n = len(arr)`, `k = n/20`, `d` = số giá trị khác nhau.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Sắp xếp + cắt | `O(n log n)` | `O(n)` |
| 2. Sắp xếp + trừ | `O(n log n)` | `O(n)` |
| 3. `heapq` | `O(n log k)` | `O(k)` |
| 4. Đếm tần suất | `O(n + d log d)` | `O(d)` |

Với `n ≤ 1000`, mọi cách đều rất nhanh; cách 1 là lựa chọn rõ ràng nhất khi đi thi. Cách 3/4 hữu ích khi `n` rất lớn và chỉ cắt một phần nhỏ.

## 11. Bộ test

1. Ba ví dụ đề (so sánh với dung sai `1e-5`).
2. Mảng toàn bằng nhau → trung bình = giá trị đó.
3. Giá trị ngoại lai (outlier) cực lớn/nhỏ đúng `k` cái → bị loại hết.
4. Nhiều giá trị trùng ở biên (ví dụ nhiều số 0) — chỉ bỏ đúng `k`.
5. Đối chiếu 7 hàm trên mảng ngẫu nhiên với `n` là bội của 20.

In [17]:
def close(a, b):
    return abs(a - b) < 1e-5

solvers = [trim_mean_slice, trim_mean_subtract, trim_mean_heap, trim_mean_counting,
           Solution().trimMean, SolutionHeap().trimMean, SolutionCounting().trimMean, SolutionStatistics().trimMean]

def check(name, arr, expected):
    for f in solvers:
        got = f(arr)
        assert close(got, expected), (name, f, got, expected)
    print(f"{name:<30} PASS ({trim_mean_slice(arr):.5f})")

for i, (arr, expected) in enumerate(examples, 1):
    check(f"ví dụ {i}", arr, expected)
check("toàn số 7 (n=20)", [7] * 20, 7.0)
check("outlier hai đầu", [0] + [50] * 18 + [10**5], 50.0)
check("outlier n=40, k=2", [0, 0] + [10] * 36 + [10**5, 10**5], 10.0)
check("nhiều 0 ở đầu, k=1", [0] * 5 + [9] * 15, (0 * 4 + 9 * 14) / 18)
check("giá trị max", [10**5] * 1000, 100000.0)

ví dụ 1                        PASS (2.00000)
ví dụ 2                        PASS (4.00000)
ví dụ 3                        PASS (4.77778)
toàn số 7 (n=20)               PASS (7.00000)
outlier hai đầu                PASS (50.00000)
outlier n=40, k=2              PASS (10.00000)
nhiều 0 ở đầu, k=1             PASS (7.00000)
giá trị max                    PASS (100000.00000)


In [18]:
import random
rng = random.Random(1619)
for _ in range(300):
    n = 20 * rng.randint(1, 10)
    arr = [rng.randint(0, rng.choice([5, 100, 10**5])) for _ in range(n)]
    ref = trim_mean_slice(arr)
    for f in solvers:
        assert close(f(arr), ref)
    a = sorted(arr)
    assert a[0] <= ref <= a[-1]                 # trung bình nằm giữa min và max
print("300 mảng ngẫu nhiên: 8 hàm khớp nhau (sai số < 1e-5)")

300 mảng ngẫu nhiên: 8 hàm khớp nhau (sai số < 1e-5)


## 12. Lỗi người mới hay gặp

### Lỗi 1: Bỏ chỉ một phần tử min và một phần tử max (bất kể n)

Đúng với `n = 20` nhưng sai với `n = 40` (phải bỏ 2 mỗi đầu).

In [19]:
def wrong_one_each(arr):
    a = sorted(arr)
    return sum(a[1:-1]) / (len(a) - 2)

print(f"sai: {wrong_one_each(ex3):.5f} | đúng: {trim_mean_slice(ex3):.5f}")

sai: 4.78947 | đúng: 4.77778


### Lỗi 2: Xóa **mọi** phần tử bằng min / max

`[x for x in arr if x != min(arr) and x != max(arr)]` bỏ hết các số trùng giá trị min/max — quá tay. Ví dụ 2 có bốn số `0`.

In [20]:
def wrong_remove_all_equal(arr):
    lo, hi = min(arr), max(arr)
    rest = [x for x in arr if x != lo and x != hi]
    return sum(rest) / len(rest)

print(f"sai: {wrong_remove_all_equal(ex2):.5f} | đúng: {trim_mean_slice(ex2):.5f}")

sai: 4.80000 | đúng: 4.00000


### Lỗi 3: Không sắp xếp trước khi cắt

`arr[k:n-k]` trên mảng **gốc** chỉ bỏ phần tử ở hai **vị trí** đầu/cuối, không phải nhỏ/lớn nhất.

In [21]:
def wrong_no_sort(arr):
    n, k = len(arr), len(arr) // 20
    return sum(arr[k:n - k]) / (n - 2 * k)

print(f"sai: {wrong_no_sort(ex2):.5f} | đúng: {trim_mean_slice(ex2):.5f}")

sai: 4.22222 | đúng: 4.00000


### Lỗi 4: Chia cho `n` thay vì số phần tử **còn lại**

In [22]:
def wrong_divide_n(arr):
    n, k = len(arr), len(arr) // 20
    a = sorted(arr)
    return sum(a[k:n - k]) / n

print(f"sai: {wrong_divide_n(ex2):.5f} | đúng: {trim_mean_slice(ex2):.5f}")

sai: 3.60000 | đúng: 4.00000


### Lỗi 5: So sánh kết quả số thực bằng `==` trong test

`trim_mean(ex3) == 4.77778` là `False` vì đáp án thật là `4.777777...`. Luôn dùng dung sai.

In [23]:
print(trim_mean_slice(ex3), trim_mean_slice(ex3) == 4.77778, close(trim_mean_slice(ex3), 4.77778))

4.777777777777778 False True


## 13. Mẹo nhận dạng pattern

- "Bỏ `p%` nhỏ nhất/lớn nhất", "trừ ngoại lai", "top-k" → **sắp xếp** hoặc **heap** (`nsmallest/nlargest`).
- Chỉ cần **tổng** của phần giữ lại → "**tổng toàn bộ − phần bỏ**".
- Giá trị nhỏ, bị chặn → **đếm tần suất** (counting) thay cho sắp xếp.
- Kết quả số thực → test bằng dung sai.
- Bài liên quan: 1491 Average Salary Excluding the Minimum and Maximum Salary, 215 Kth Largest Element, 347 Top K Frequent Elements, 2099 Find Subsequence of Length K With the Largest Sum.

## 14. Biến thể

### 14.1. Trung bình cắt `p%` tổng quát (làm tròn xuống số phần tử bỏ)

In [24]:
def trimmed_mean_percent(arr, p):
    n = len(arr)
    k = n * p // 100
    a = sorted(arr)
    if n - 2 * k <= 0:
        raise ValueError("cắt hết dữ liệu rồi!")
    return sum(a[k:n - k]) / (n - 2 * k)

print(trimmed_mean_percent(ex2, 5), trimmed_mean_percent(ex2, 10), trimmed_mean_percent(ex2, 0))

4.0 4.0 4.1


### 14.2. LeetCode 1491 — trung bình lương bỏ min và max (mỗi giá trị khác nhau)

In [25]:
def average_salary(salary):
    return (sum(salary) - min(salary) - max(salary)) / (len(salary) - 2)

print(average_salary([4000, 3000, 1000, 2000]))   # 2500.0

2500.0


### 14.3. Trung vị (median) — trường hợp "cắt" cực hạn

Trung vị là phần tử giữa sau khi sắp xếp (hoặc trung bình hai phần tử giữa khi `n` chẵn). Có thể coi là trimmed mean với gần 50% mỗi đầu.

In [26]:
def median(arr):
    a = sorted(arr)
    n = len(a)
    mid = n // 2
    return a[mid] if n % 2 == 1 else (a[mid - 1] + a[mid]) / 2

print(median(ex2), statistics.median(ex2))

5.0 5.0


## 15. Bài tập tự luyện (TODO)

**Bài 1 — tính tay.** Với `n = 60`, phải bỏ bao nhiêu phần tử mỗi đầu? Còn lại bao nhiêu?

**Bài 2 — đoán output.** `sorted([3, 1, 2])[1:-1]` và `[5, 6, 7][0:-0]` in ra gì?

### Bài 3 — Điền TODO: lời giải chính

In [27]:
def exercise_trim_mean(arr: list[int]) -> float:
    # TODO 1: n, k = len(arr), len(arr) // 20
    # TODO 2: a = sorted(arr)
    # TODO 3: trả trung bình của a[k:n-k]
    pass

# assert abs(exercise_trim_mean([6,2,7,5,1,2,0,3,10,2,5,0,5,5,0,8,7,6,8,0]) - 4.0) < 1e-5

### Bài 4 — Tự viết cách dùng `heapq` (không sắp xếp toàn bộ)

In [28]:
def exercise_trim_mean_heap(arr: list[int]) -> float:
    # TODO: tổng toàn bộ - tổng k nhỏ nhất - tổng k lớn nhất
    pass

# assert abs(exercise_trim_mean_heap(ex3) - 4.77778) < 1e-5

### Bài 5 — Biến thể: trung bình lương bỏ min/max (LeetCode 1491) không dùng `min`/`max`/`sorted`

In [29]:
def exercise_average_salary(salary: list[int]) -> float:
    # TODO: một vòng lặp tìm lo, hi và tổng
    pass

# assert exercise_average_salary([4000, 3000, 1000, 2000]) == 2500.0

### Bài 6 — Mini challenge: trimmed mean với mảng đếm cố định `0..10^5`

Không dùng `sorted`, không dùng `Counter`: tạo `cnt = [0] * (10**5 + 1)` rồi bỏ `k` phần tử mỗi đầu.

In [30]:
def exercise_trim_mean_array_count(arr: list[int]) -> float:
    # TODO
    pass

# assert abs(exercise_trim_mean_array_count(ex2) - 4.0) < 1e-5

## 16. Gợi ý 3 mức

**Bài 1:** M1: 5% của 60. M2: `60 // 20`. M3: còn `60 − 2·3`.

**Bài 2:** M1: sorted trước. M2: `[1, 2, 3][1:-1]`. M3: `-0` chính là `0`.

**Bài 3:** M1: làm theo mục 3. M2: chia cho `n - 2*k`. M3: `return sum(a[k:n-k]) / (n - 2*k)`.

**Bài 4:** M1: `heapq.nsmallest(k, arr)`. M2: tương tự `nlargest`. M3: `(sum(arr) - lo - hi) / (n - 2*k)`.

**Bài 5:** M1: khởi tạo `lo = hi = salary[0]`. M2: cập nhật trong vòng lặp. M3: `(total - lo - hi) / (n - 2)`.

**Bài 6:** M1: `cnt[v] += 1`. M2: duyệt `v` từ 0 lên, mỗi bước bỏ `min(left, cnt[v])`. M3: duyệt từ `10**5` xuống cho đầu lớn.

## 17. Đáp án tham khảo

**Bài 1:** `k = 3`, còn lại `54`.

**Bài 2:** `[2]` và `[]`.

In [31]:
def answer_trim_mean(arr):
    n, k = len(arr), len(arr) // 20
    a = sorted(arr)
    return sum(a[k:n - k]) / (n - 2 * k)

def answer_trim_mean_heap(arr):
    n, k = len(arr), len(arr) // 20
    return (sum(arr) - sum(heapq.nsmallest(k, arr)) - sum(heapq.nlargest(k, arr))) / (n - 2 * k)

def answer_average_salary(salary):
    lo = hi = total = salary[0]
    for s in salary[1:]:
        total += s
        if s < lo:
            lo = s
        if s > hi:
            hi = s
    return (total - lo - hi) / (len(salary) - 2)

def answer_trim_mean_array_count(arr):
    n, k = len(arr), len(arr) // 20
    MAXV = 10**5
    cnt = [0] * (MAXV + 1)
    for v in arr:
        cnt[v] += 1
    def drop(values):
        left, s = k, 0
        for v in values:
            if cnt[v]:
                t = min(left, cnt[v])
                s += t * v
                left -= t
                if left == 0:
                    break
        return s
    return (sum(arr) - drop(range(MAXV + 1)) - drop(range(MAXV, -1, -1))) / (n - 2 * k)

assert 60 // 20 == 3 and 60 - 2 * 3 == 54
assert sorted([3, 1, 2])[1:-1] == [2] and [5, 6, 7][0:-0] == []
assert close(answer_trim_mean(ex2), 4.0)
assert close(answer_trim_mean_heap(ex3), 4.77778)
assert answer_average_salary([4000, 3000, 1000, 2000]) == 2500.0
assert close(answer_trim_mean_array_count(ex2), 4.0)
for _ in range(100):
    n = 20 * rng.randint(1, 5)
    arr = [rng.randint(0, 10**5) for _ in range(n)]
    ref = trim_mean_slice(arr)
    assert close(answer_trim_mean(arr), ref) and close(answer_trim_mean_heap(arr), ref)
    assert close(answer_trim_mean_array_count(arr), ref)
print("Đáp án tham khảo: PASS toàn bộ!")

Đáp án tham khảo: PASS toàn bộ!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def trimMean(self, arr: list[int]) -> float:
        n = len(arr)
        k = n // 20                      # 5% (n chia hết cho 20)
        a = sorted(arr)                  # nhỏ ở đầu, lớn ở cuối
        return sum(a[k:n - k]) / (n - 2 * k)
```

**Câu thần chú:** *"Sắp xếp, cắt k = n/20 mỗi đầu, chia cho phần còn lại."*

**Checklist trước khi submit:**
- [ ] `k = n // 20` (không cố định là 1)?
- [ ] Đã **sắp xếp** trước khi cắt?
- [ ] Chỉ bỏ **đúng `k`** phần tử mỗi đầu (không xóa mọi số trùng min/max)?
- [ ] Chia cho `n - 2k`, dùng `/` để ra `float`?
- [ ] Test số thực bằng dung sai, không dùng `==`?
- [ ] Tên method đúng `trimMean`?